# Customer Churn Analysis

**Objective:** Calculate churn rate, Customer Lifetime Value (CLV) and MRR loss, find the behavioural drivers of churn, and recommend actions.

**Input:** `data/customer_churn_clean.csv` – the output of the data-cleaning task (600 customers, 0 missing values, 0 duplicates).  
**Outputs:** charts in `charts/` and `metrics.json` (used by the PDF report).

In [1]:
import pandas as pd, numpy as np, json, os
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid', font_scale=1.0)
os.makedirs('charts', exist_ok=True)
PAL = {'Yes':'#E4572E','No':'#4C78A8'}
df = pd.read_csv('data/customer_churn_clean.csv')
df['tenure_band'] = pd.cut(df.tenure_months, [-1,12,24,48,72], labels=['0-12','13-24','25-48','49-72'])
M = {}   # metrics for the report
print(df.shape, df.isna().sum().sum(), 'missing')

(600, 16) 0 missing


## 1. Validate the data
The task says to validate before drawing conclusions. The cleaned file passes the checks below (ID unique, no nulls, sensible ranges, monthly price inside each plan's band).

In [2]:
assert df.customer_id.is_unique and df.isna().sum().sum() == 0
assert df.age.between(18,100).all() and df.tenure_months.between(0,120).all()
assert df.churn.isin(['Yes','No']).all()
print(df.groupby('subscription_type').monthly_charges_inr.agg(['min','max']).round(0))

                      min     max
subscription_type                
Basic               364.0   484.0
Pro                1064.0  1184.0
Standard            664.0   783.0


## 2. Baseline metrics: churn rate, MRR loss, CLV

- **Churn rate** = churned customers ÷ all customers.
- **MRR loss** = monthly charges of churned customers; as a share of total MRR.
- **ARPU** = average monthly charge per customer.
- **Monthly churn rate** = churned customers ÷ total customer-months observed (tenure). Using customer-months, not customers, is the correct way to get a *per-month* rate because customers have been observed for very different lengths of time.
- **CLV** = ARPU ÷ monthly churn rate (ARPU × expected lifetime in months, lifetime = 1 ÷ monthly churn). This assumes a constant monthly churn rate, so treat it as an estimate.

In [3]:
n = len(df); churned = int(df.churn_flag.sum())
M['customers']=n; M['churned']=churned; M['churn_rate']=churned/n
M['mrr_total']=df.monthly_charges_inr.sum(); M['mrr_lost']=df.loc[df.churn_flag==1,'monthly_charges_inr'].sum()
M['mrr_loss_pct']=M['mrr_lost']/M['mrr_total']; M['arr_lost']=M['mrr_lost']*12
M['arpu']=df.monthly_charges_inr.mean()
M['monthly_churn']=churned/df.tenure_months.sum()
M['lifetime_months']=1/M['monthly_churn']
M['clv']=M['arpu']*M['lifetime_months']
M['avg_tenure_churned']=df.loc[df.churn_flag==1,'tenure_months'].mean()
M['avg_tenure_retained']=df.loc[df.churn_flag==0,'tenure_months'].mean()
for k,v in M.items(): print(f'{k:22s} {v:,.3f}')

customers              600.000
churned                247.000
churn_rate             0.412
mrr_total              455,777.640
mrr_lost               187,319.750
mrr_loss_pct           0.411
arr_lost               2,247,837.000
arpu                   759.629
monthly_churn          0.012
lifetime_months        86.336
clv                    65,583.388
avg_tenure_churned     32.543
avg_tenure_retained    37.640


## 3. Churn by contract type, tenure and payment method

In [4]:
def seg(col):
    g = df.groupby(col, observed=True).agg(customers=('churn_flag','size'), churned=('churn_flag','sum'),
        churn_rate=('churn_flag','mean'), mrr_lost=('monthly_charges_inr', lambda s: s[df.loc[s.index,'churn_flag']==1].sum()),
        arpu=('monthly_charges_inr','mean'), tenure_months=('tenure_months','sum'))
    g['monthly_churn'] = g.churned/g.tenure_months
    g['clv'] = g.arpu/g.monthly_churn
    return g.drop(columns='tenure_months')
contract = seg('contract_type').loc[['Month-to-Month','One Year','Two Year']]
tenure   = seg('tenure_band')
payment  = seg('payment_method').sort_values('churn_rate', ascending=False)
plan     = seg('subscription_type')
M['contract']=contract.round(4).reset_index().to_dict('records')
M['tenure']=tenure.round(4).reset_index().astype({'tenure_band':str}).to_dict('records')
M['payment']=payment.round(4).reset_index().to_dict('records')
M['plan']=plan.round(4).reset_index().to_dict('records')
display(contract.round(3)); display(tenure.round(3)); display(payment.round(3)); display(plan.round(3))

,customers,churned,churn_rate,mrr_lost,arpu,monthly_churn,clv
contract_type,,,,,,,
Month-to-Month,316,189,0.598,144427.00,773.845,0.017,44756.087
One Year,164,33,0.201,25640.62,759.337,0.006,137578.094
Two Year,120,25,0.208,17252.13,722.594,0.006,127610.130


,customers,churned,churn_rate,mrr_lost,arpu,monthly_churn,clv
tenure_band,,,,,,,
0-12,104,56,0.538,40389.95,742.296,0.086,8615.936
13-24,110,50,0.455,41615.48,754.199,0.024,30891.980
25-48,188,72,0.383,54931.84,765.756,0.011,71693.862
49-72,198,69,0.348,50382.48,765.934,0.006,131940.473


,customers,churned,churn_rate,mrr_lost,arpu,monthly_churn,clv
payment_method,,,,,,,
UPI,142,64,0.451,50199.33,759.907,0.013,59367.771
Wallet,161,72,0.447,52631.31,757.444,0.013,59007.019
Card,157,59,0.376,42493.35,734.980,0.010,70832.116
Net Banking,140,52,0.371,41995.76,789.503,0.010,76369.233


,customers,churned,churn_rate,mrr_lost,arpu,monthly_churn,clv
subscription_type,,,,,,,
Basic,211,86,0.408,36683.23,424.134,0.012,35632.202
Pro,210,85,0.405,95722.86,1127.260,0.012,97687.038
Standard,179,76,0.425,54913.66,723.803,0.011,64132.737


## 4. Other behavioural drivers

In [5]:
late = df.groupby('late_payments').churn_flag.agg(['mean','size']).round(3)
tick = df.groupby('support_tickets').churn_flag.agg(['mean','size']).round(3)
ap   = df.groupby('auto_pay').churn_flag.mean().round(3)
reg  = df.groupby('region').churn_flag.mean().round(3)
M['late']=late.reset_index().to_dict('records'); M['tickets']=tick.reset_index().to_dict('records')
M['autopay']={str(k):v for k,v in ap.items()}
corr = df[['age','tenure_months','monthly_charges_inr','support_tickets','late_payments','churn_flag']].corr()['churn_flag'].drop('churn_flag').round(3)
M['corr']=corr.to_dict()
display(late); display(tick); display(ap); display(corr)

,mean,size
late_payments,,
0,0.244,172
1,0.330,188
2,0.551,156
3,0.690,71
4,0.615,13


,mean,size
support_tickets,,
0,0.267,146
1,0.393,135
2,0.429,133
3,0.496,113
4,0.481,54
5,0.786,14
6,1.000,5


auto_pay
False    0.412
True     0.411
Name: churn_flag, dtype: float64

age                   -0.012
tenure_months         -0.120
monthly_charges_inr   -0.004
support_tickets        0.212
late_payments          0.311
Name: churn_flag, dtype: float64

**Reading the numbers:** contract type, late payments and support tickets move churn a lot; tenure moves it moderately (newer customers churn more). Age, plan, price, auto-pay and region show little or no relationship.

## 5. Charts

In [6]:
def barplot(data, x, title, fname, order=None, rot=0, xlabel=None):
    fig, ax = plt.subplots(figsize=(6.4,3.8))
    d = data.reset_index() if x not in data.columns else data
    sns.barplot(data=d, x=x, y=d.columns[d.columns.str.contains('churn_rate')][0], order=order, color='#4C78A8', ax=ax)
    ax.axhline(M['churn_rate'], color='#E4572E', ls='--', lw=1.5, label=f"Overall {M['churn_rate']:.1%}")
    for p, n_ in zip(ax.patches, (d.set_index(x).loc[[t.get_text() for t in ax.get_xticklabels()],'customers'])):
        ax.annotate(f'{p.get_height():.1%}\n(n={n_})', (p.get_x()+p.get_width()/2, p.get_height()), ha='center', va='bottom', fontsize=9)
    ax.set_ylim(0, d.filter(like='churn_rate').max().iloc[0]*1.25); ax.set_ylabel('Churn rate'); ax.set_xlabel(xlabel or ''); ax.set_title(title, fontweight='bold')
    ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0)); plt.xticks(rotation=rot); ax.legend(loc='upper right')
    plt.tight_layout(); plt.savefig(f'charts/{fname}.png', dpi=200); plt.show()

barplot(contract.reset_index(), 'contract_type', 'Churn rate by contract type', '1_contract', xlabel='Contract type')
barplot(tenure.reset_index().astype({'tenure_band':str}), 'tenure_band', 'Churn rate by tenure (months)', '2_tenure', xlabel='Tenure band (months)')
barplot(payment.reset_index(), 'payment_method', 'Churn rate by payment method', '3_payment', xlabel='Payment method')

In [7]:
# Box plots: how churned vs retained customers differ
fig, axes = plt.subplots(1, 4, figsize=(11,3.8))
for ax, col, t in zip(axes, ['tenure_months','late_payments','support_tickets','monthly_charges_inr'],
                      ['Tenure (months)','Late payments','Support tickets','Monthly charge (INR)']):
    sns.boxplot(data=df, x='churn', y=col, order=['No','Yes'], palette=PAL, hue='churn', legend=False, ax=ax, width=.55)
    ax.set_title(t, fontsize=10, fontweight='bold'); ax.set_xlabel('Churned?'); ax.set_ylabel('')
plt.suptitle('Churned vs retained customers', fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig('charts/4_boxplots.png', dpi=200, bbox_inches='tight'); plt.show()

In [8]:
# Heatmaps: churn rate across two dimensions at once
def heat(index, cols, title, fname, order_i=None, order_c=None):
    rate = df.pivot_table(index=index, columns=cols, values='churn_flag', aggfunc='mean', observed=True)
    cnt  = df.pivot_table(index=index, columns=cols, values='churn_flag', aggfunc='size', observed=True)
    if order_i: rate, cnt = rate.loc[order_i], cnt.loc[order_i]
    if order_c: rate, cnt = rate[order_c], cnt[order_c]
    lab = rate.map(lambda v: f'{v:.0%}') + '\n(n=' + cnt.astype(int).astype(str) + ')'
    fig, ax = plt.subplots(figsize=(6.4,3.6))
    sns.heatmap(rate, annot=lab, fmt='', cmap='YlOrRd', vmin=0, vmax=.8, cbar_kws={'label':'Churn rate'}, ax=ax, linewidths=.5)
    ax.set_title(title, fontweight='bold'); ax.set_xlabel(''); ax.set_ylabel('')
    plt.tight_layout(); plt.savefig(f'charts/{fname}.png', dpi=200); plt.show()
ct = ['Month-to-Month','One Year','Two Year']
heat('contract_type','tenure_band','Churn rate: contract type × tenure','5_heat_contract_tenure', order_i=ct)
heat('contract_type','payment_method','Churn rate: contract type × payment method','6_heat_contract_payment', order_i=ct)

df['late_band']=pd.cut(df.late_payments,[-1,0,1,4],labels=['0','1','2+'])
heat('late_band','contract_type','Churn rate: late payments × contract type','7_heat_late_contract', order_c=ct)

In [9]:
# Late payments and support tickets
fig, axes = plt.subplots(1,2, figsize=(10,3.6))
for ax,(t_,col) in zip(axes,[('Late payments','late_payments'),('Support tickets','support_tickets')]):
    g = df.groupby(col).churn_flag.agg(['mean','size']).reset_index()
    sns.barplot(data=g, x=col, y='mean', color='#4C78A8', ax=ax)
    for p, n_ in zip(ax.patches, g['size']): ax.annotate(f'{p.get_height():.0%}\n(n={n_})', (p.get_x()+p.get_width()/2,p.get_height()), ha='center', va='bottom', fontsize=8)
    ax.axhline(M['churn_rate'], color='#E4572E', ls='--'); ax.set_ylim(0,1.2); ax.set_ylabel('Churn rate'); ax.set_xlabel(t_)
    ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0)); ax.set_title(f'Churn rate by {t_.lower()}', fontweight='bold')
plt.tight_layout(); plt.savefig('charts/8_late_tickets.png', dpi=200); plt.show()

# MRR lost by contract
fig, ax = plt.subplots(figsize=(6.4,3.6))
m = contract.reset_index()
sns.barplot(data=m, x='contract_type', y='mrr_lost', color='#E4572E', ax=ax)
for p in ax.patches: ax.annotate(f'₹{p.get_height():,.0f}', (p.get_x()+p.get_width()/2,p.get_height()), ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Monthly revenue lost (INR)'); ax.set_xlabel(''); ax.set_title('MRR lost to churn by contract type', fontweight='bold'); ax.set_ylim(0, m.mrr_lost.max()*1.15)
plt.tight_layout(); plt.savefig('charts/9_mrr_lost.png', dpi=200); plt.show()

## 6. What-if: how much revenue could contract conversion save?
Scenario: move **25 % of month-to-month customers** onto one-year contracts, whose churn rate is about 20 % instead of about 60 %. This is an estimate, not a guarantee: customers who choose long contracts may already be more loyal, so the real effect could be smaller.

In [10]:
m2m = contract.loc['Month-to-Month']; one = contract.loc['One Year']
share = .25
moved = m2m.customers*share
saved_customers = moved*(m2m.churn_rate-one.churn_rate)
avg_m = df.loc[df.contract_type=='Month-to-Month','monthly_charges_inr'].mean()
M['scenario']={'share':share,'moved':moved,'saved_customers':saved_customers,'mrr_saved':saved_customers*avg_m,
               'pct_of_lost':saved_customers*avg_m/M['mrr_lost']}
# Customers with 2+ late payments
risk = df[df.late_payments>=2]
M['late2_n']=len(risk); M['late2_rate']=risk.churn_flag.mean(); M['late2_share_of_churn']=risk.churn_flag.sum()/churned
M['late2_mrr_lost']=risk.loc[risk.churn_flag==1,'monthly_charges_inr'].sum()
t3 = df[df.support_tickets>=3]; M['t3_n']=len(t3); M['t3_rate']=t3.churn_flag.mean()
newc = df[df.tenure_months<=12]; M['new_n']=len(newc); M['new_rate']=newc.churn_flag.mean()
json.dump(M, open('metrics.json','w'), indent=1, default=float)
M['scenario'], M['late2_rate'], M['late2_share_of_churn'], M['t3_rate'], M['new_rate']

({'share': 0.25,
  'moved': np.float64(79.0),
  'saved_customers': np.float64(31.353658536585364),
  'mrr_saved': np.float64(24262.87387465267),
  'pct_of_lost': np.float64(0.12952651215182953)},
 np.float64(0.5958333333333333),
 np.float64(0.5789473684210527),
 np.float64(0.5268817204301075),
 np.float64(0.5384615384615384))

## 7. Key findings & recommendations

1. **Contract type is the biggest driver.** Month-to-month customers churn at ~60 % versus ~20 % on one- and two-year contracts, and account for ~3 of every 4 churned customers.  
   → Offer discounts/credits to move month-to-month customers onto annual plans, especially after month 3.
2. **Late payments are the strongest behavioural warning sign.** Churn climbs from ~24 % (no late payments) to ~55 % (2 late payments) and ~69 % (3).  
   → Trigger a retention call or payment-reminder flow after the *first* late payment, offer flexible due dates.
3. **Support tickets signal frustration.** Churn rises from ~27 % (0 tickets) to ~50 % (3 tickets) and ~79 % (5+).  
   → Escalate customers with 3+ tickets to a senior agent and follow up after resolution.
4. **Early life is the riskiest.** Customers in their first year churn ~54 % vs ~35 % after four years.  
   → Build an onboarding programme for the first 12 months.
5. **UPI and Wallet users churn more** (~45 %) than Card/Net Banking (~37 %).  
   → Push card/net-banking auto-debit with a small incentive, and fix payment failures on UPI/Wallet.
6. **Not drivers here:** age, price, plan type, auto-pay and region show little or no effect, so price cuts or plan changes are unlikely to be the best lever.

*Caveat:* this is observational data (correlation, not proof of cause) and the CLV uses a constant-churn assumption.